## Read the Bronze ERP customer table 

In [0]:
from pyspark.sql import functions as F

erp_cust = spark.table(
    "e2e_project.bronze.erp_cust_az12"
)

display(erp_cust)

In [0]:
erp_cust.printSchema()
print("Rows:", erp_cust.count())

CID   = customer identifier
BDATE = birth date
GEN   = gender

## Inspect the customer IDs

In [0]:
display(
    erp_cust.select("CID").limit(50)
)

CRM uses the shorter version, so Silver should normalize the ERP identifier.

## Check how many IDs contain NAS

In [0]:
display(
    erp_cust.filter(
        F.col("CID").startswith("NAS")
    )
)

detect

   ↓

understand

   ↓

transform


## Remove the NAS prefix

Create the Silver DataFrame:

In [0]:
silver_erp_cust = erp_cust.withColumn(
    "CID",
    F.when(
        F.col("CID").startswith("NAS"),
        F.substring(F.col("CID"), 4, 100)
    )
    .otherwise(F.col("CID"))
)

## Rename the ID now

In [0]:
silver_erp_cust = silver_erp_cust.withColumnRenamed(
    "CID",
    "customer_id"
)

## Inspect birth dates

In [0]:
display(
    silver_erp_cust.select("BDATE")
)

In [0]:
silver_erp_cust.printSchema()

If BDATE isn't already a Spark date, convert it:

In [0]:
silver_erp_cust = silver_erp_cust.withColumn(
    "BDATE",
    F.to_date(F.col("BDATE"))
)

## Look for impossible birth dates

In [0]:
display(
    silver_erp_cust.filter(
        F.col("BDATE") > F.current_date()
    )
)

In [0]:
silver_erp_cust = silver_erp_cust.withColumn(
    "BDATE",
    F.when(
        F.col("BDATE") > F.current_date(),
        F.lit(None).cast("date")
    )
    .otherwise(F.col("BDATE"))
)

## Rename birth date

In [0]:
silver_erp_cust = silver_erp_cust.withColumnRenamed(
    "BDATE",
    "birth_date"
)

CID      → customer_id

BDATE    → birth_date

## Inspect gender values


In [0]:
display(
    silver_erp_cust
    .groupBy("GEN")
    .count()
)

## Standardize gender

In [0]:
silver_erp_cust = silver_erp_cust.withColumn(
    "GEN",
    F.when(
        F.upper(F.trim(F.col("GEN"))).isin("F", "FEMALE"),
        "Female"
    )
    .when(
        F.upper(F.trim(F.col("GEN"))).isin("M", "MALE"),
        "Male"
    )
    .otherwise("n/a")
)

In [0]:
silver_erp_cust = silver_erp_cust.withColumnRenamed(
    "GEN",
    "gender"
)

## Inspect the cleaned table

In [0]:
display(silver_erp_cust)

In [0]:
silver_erp_cust.printSchema()

## Validate customer IDs

In [0]:
display(
    silver_erp_cust.filter(
        F.col("customer_id").isNull()
    )
)

In [0]:
display(
    silver_erp_cust.filter(
        F.col("customer_id").startswith("NAS")
    )
)

## Check duplicate customer IDs

In [0]:
display(
    silver_erp_cust
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

## Validate gender 

In [0]:
display(
    silver_erp_cust
    .groupBy("gender")
    .count()
)

## Validate birth dates again

In [0]:
display(
    silver_erp_cust.filter(
        F.col("birth_date") > F.current_date()
    )
)

## Write to Silver

In [0]:
(
    silver_erp_cust.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "e2e_project.silver.erp_cust_az12"
    )
)